identification of Dopimanergic Neurons in Substantia Nigra using ASAP CRN scnRNA cohort data.


In [2]:
import scvi
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import sys, subprocess, importlib, warnings, math, os

from pathlib import Path

In [3]:
##
# pip3 install -U scvi-tools[cuda]  # gets jax and jaxlib, updates cuda
# pip3 install -U scib-metrics
#
#
#

## 1. Workspace Setup

### 1.1 Set dataset paths
In this example, we are working with the **PMDBS single‑cell RNA‑seq cohort** dataset:

- **Workflow** → `pmdbs_sc_rnaseq`  
- **Team** → `cohort`  
- **Source** → `pmdbs`  
- **Type** → `sc-rnaseq`  

These components are combined to construct the bucket and dataset names.  
We then set the path to the **cohort analysis outputs** and preview the available files.


In [4]:
#set general folder paths
HOME = Path.home()
WS_ROOT = HOME / "workspace"
DATA_DIR = WS_ROOT / "data"
WS_FILES = WS_ROOT / "ws_files"

if not WS_ROOT.exists():
    print(f"{WS_ROOT} doesn't exist. We need to remount our resources")
    !wb resource mount    

print("Home directory:     ", HOME)
print("Workspace root:     ", WS_ROOT)
print("Data directory:     ", DATA_DIR)
print("ws_files directory: ", WS_FILES)

print("\nContents of workspace root:")
for p in WS_ROOT.glob("*"):
    print(" -", p.name, "/" if p.is_dir() else "")

Home directory:      /home/ergonyc
Workspace root:      /home/ergonyc/workspace
Data directory:      /home/ergonyc/workspace/data
ws_files directory:  /home/ergonyc/workspace/ws_files

Contents of workspace root:
 - ws_files /
 - 01_PMDBS_scRNAseq_Datasets /
 - release_resources /
 - README 
 - 2025_CRN_CM_Workshop_Resources_03122025 /


In [5]:
## Build and set path to desired dataset

DATASETS_PATH = WS_ROOT / "01_PMDBS_scRNAseq"

workflow       = "pmdbs_sc_rnaseq"
dataset_team   = "cohort"
dataset_source = "pmdbs"
dataset_type   = "sc-rnaseq"

bucket_name  = f"asap-curated-{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name = f"asap-{dataset_team}-{dataset_source}-{dataset_type}"

dataset_path = DATASETS_PATH / bucket_name / workflow
print("Dataset Path:", dataset_path)

# Build the folder path to the cohort analysis directory
cohort_analysis_path = dataset_path / "cohort_analysis"

# Preview the directory contents
print("Contents of cohort_analysis:")
!ls {cohort_analysis_path}

Dataset Path: /home/ergonyc/workspace/01_PMDBS_scRNAseq/asap-curated-cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq
Contents of cohort_analysis:
ls: cannot access '/home/ergonyc/workspace/01_PMDBS_scRNAseq/asap-curated-cohort-pmdbs-sc-rnaseq/pmdbs_sc_rnaseq/cohort_analysis': No such file or directory


In [6]:
# Define a local path for workshop files
local_data_path = WS_FILES / "data"

# Create the directory if it doesn't already exist
if not local_data_path.exists():
    local_data_path.mkdir(parents=True)

print(f"Local data directory ready at: {local_data_path}")

Local data directory ready at: /home/ergonyc/workspace/ws_files/data


### Copy Data Locally

We now bring in the curated dataset files:

- **`asap-cohort.final_metadata.csv`** → cell‑level metadata table
- **`asap-cohort.final.h5ad`** → full AnnData object containing HVG expression data and annotations  

We copy these files into our local `pilot_workshop_files` directory (if not already present) and load them into memory.

The metadata CSV is read into a Pandas dataframe, while the `.h5ad` file is loaded as an AnnData object in backed mode.

In [6]:
# # Downloading obs field (cell metadata)
# # Define the expected local path for the metadata file.
# cell_metadata_local_path = local_data_path / f"asap-{dataset_team}.final_metadata.csv"\

# # Check if the metadata file already exists locally.
# if not cell_metadata_local_path.exists():
#     # Construct the original path where the metadata file is stored.
#     cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final_metadata.csv"

#     # Use a shell command (`cp`) to copy the file from the original location
#     # into the local workshop_files directory for analysis.
#     !cp {cell_metadata_og_path} {cell_metadata_local_path}

In [7]:
# Downloading the anndata object
# Define the expected local path
adata_local_path = local_data_path / f"asap-{dataset_team}.final.h5ad"

# # Check if the adata file already exists locally.
# if not adata_local_path.exists():
#     # Construct the original path where the metadata file is stored.
#     adata_cell_metadata_og_path = cohort_analysis_path / f"asap-{dataset_team}.final.h5ad"

#     # Use a shell command (`cp`) to copy the file from the original location
#     # into the local workshop_files directory for analysis.
#     !cp {adata_cell_metadata_og_path} {adata_local_path}

# load the adata object
adata = sc.read_h5ad(adata_local_path, backed="r")
adata

AnnData object with n_obs × n_vars = 3578056 × 3000 backed at '/home/ergonyc/workspace/ws_files/data/asap-cohort.final.h5ad'
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'total_counts', 'total_counts_rb', 'pct_counts_rb', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'sample', 'batch', 'team', 'dataset', 'batch_id', 'S_score', 'G2M_score', 'phase', 'cell_type', 'phenotype', 'rho', 'prob', 'class_name', 'subclass_name', 'supertype_name', '_scvi_batch', '_scvi_labels', 'C_scANVI', 'leiden_res_0.05', 'leiden_res_0.10', 'leiden_res_0.20', 'leiden_res_0.40'
    var: 'feature_type', 'genome', 'gene_id', 'mt', 'rb'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'estimator', 'fraction_data_used_for_testing', 'learning_curve_learning_rate_epoch', 'learning_curve_test_epoch', 'learning_curve_train_epoch', 'leiden_res_0.05', 'leiden_res_0.10', 'leiden_res_0.20', 'leiden_res_0.40', 'log1p', 'neighbors', 'pca', 'scrublet', 'target_

### Metadata Access & Integration 

In [7]:
#Define metadata folder path
ds_metadata_path = WS_FILES / "metadata/cohort-pmdbs-sc-rnaseq/metadata"

#preview contents
!ls {ds_metadata_path} 

ASSAY_RNAseq.csv  collection_version  PMDBS.csv     STUDY.csv
cde_version	  CONDITION.csv       PROTOCOL.csv  SUBJECT.csv
CLINPATH.csv	  DATA.csv	      SAMPLE.csv


In [8]:
# Sample-level metadata
SAMPLE = pd.read_csv(ds_metadata_path / "SAMPLE.csv", index_col=0)
# Subject-level metadata
SUBJECT = pd.read_csv(ds_metadata_path / "SUBJECT.csv", index_col=0)
#  Brain-sample metadata
PMDBS = pd.read_csv(ds_metadata_path / "PMDBS.csv", index_col=0)
# Experimental condition metadata
CONDITION = pd.read_csv(ds_metadata_path / "CONDITION.csv", index_col=0)

# Select Relevant Columns
sample_cols = [
    "ASAP_sample_id",
    "ASAP_subject_id",
    "ASAP_team_id",
    "ASAP_dataset_id",
    "replicate",
    "condition_id",
    "age_at_collection",
]
subject_cols = [
    "ASAP_subject_id",
    "source_subject_id",
    "sex",
    "primary_diagnosis",
]
pmdbs_cols = [
    "ASAP_sample_id",
    "brain_region",
    "region_level_1",
    "region_level_2",
    "region_level_3",
]
condition_cols = [
    "condition_id",
    "intervention_name",
    "intervention_id",
    "protocol_id",
]

In [9]:
CONDITION

,ASAP_team_id,ASAP_dataset_id,condition_id,intervention_name,intervention_id,protocol_id,intervention_aux_table
0,TEAM_HAFLER,DS_PMDBS_0002,Control,Case-Control,NaN,NaN,NaN
1,TEAM_HAFLER,DS_PMDBS_0002,PD,Case-Control,NaN,NaN,NaN
2,TEAM_LEE,DS_PMDBS_0001,Control,Case-Control,NaN,NaN,NaN
3,TEAM_LEE,DS_PMDBS_0001,PD,Case-Control,NaN,NaN,NaN
4,TEAM_SCHERZER,DS_PMDBS_0005,PD,Case-Control,Case,NaN,NaN
5,TEAM_SCHERZER,DS_PMDBS_0005,Control,Case-Control,Control,NaN,NaN
6,TEAM_SCHERZER,DS_PMDBS_0005,Prodromal,Case-Control,Other,NaN,NaN
7,TEAM_HARDY,DS_PMDBS_0003,PD,Case-Control,NaN,NaN,NaN
8,TEAM_HARDY,DS_PMDBS_0003,Control,Case-Control,NaN,NaN,NaN
9,TEAM_JAKOBSSON,DS_PMDBS_0004,PD,Case-Control,Case,NaN,NaN


In [12]:
# patch TEAM_SULZER condition_id.
gp2_phenotype_mapper = {
    "no_pd_nor_other_neurological_disorder": "Control",
    "alzheimers_disease": "Control",
    "other_neurological_disorder": "Control",
    "idiopathic_pd": "PD",
    "Control": "Control",
    "PD": "PD",
    "Prodromal": "Prodromal",
}


CONDITION["condition_id"] = CONDITION["condition_id"].map(gp2_phenotype_mapper)

CONDITION["intervention_id"] = CONDITION["condition_id"].map(
    {"Control": "Control", "PD": "PD", "Prodromal": "Case"}
)

# drop duplicates
CONDITION = CONDITION.drop_duplicates()

/tmp/ipykernel_61652/3903134728.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CONDITION["condition_id"] = CONDITION["condition_id"].map(gp2_phenotype_mapper)
/tmp/ipykernel_61652/3903134728.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CONDITION["intervention_id"] = CONDITION["condition_id"].map(


In [13]:
SAMPLE.condition_id.value_counts()

condition_id
PD           294
Control      189
Prodromal     31
Name: count, dtype: int64

In [14]:
# i don't think merge is right here...
df = pd.merge(
    SAMPLE[sample_cols].copy(),
    CONDITION[condition_cols].copy(),
    on=["condition_id"],
    how="left",  # keep all SAMPLE rows, add CONDITION info
    validate="many_to_many",  # each SAMPLE row maps to one CONDITION row
)

In [12]:
df

,ASAP_sample_id,ASAP_subject_id,ASAP_team_id,ASAP_dataset_id,replicate,condition_id,age_at_collection,intervention_name,intervention_id,protocol_id
0,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN
1,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN
2,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN
3,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN
4,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN
...,...,...,...,...,...,...,...,...,...,...
2924,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN
2925,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN
2926,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN
2927,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN


In [13]:
df = pd.merge(
    df,
    SUBJECT[subject_cols],
    on=["ASAP_subject_id"],
    how="left",  # keep all SAMPLE rows, add SUBJECT info
    validate="many_to_many",  # each SUBJECT row maps to multiple SAMPLE rows
)

# Merge in brain-region information
df = pd.merge(
    df, PMDBS[pmdbs_cols], on=["ASAP_sample_id"], how="left", validate="many_to_many"
)

# create unique sample identifier
df["sample"] = df["ASAP_sample_id"] + "_" + df["replicate"]

In [14]:
df.head()

,ASAP_sample_id,ASAP_subject_id,ASAP_team_id,ASAP_dataset_id,replicate,condition_id,age_at_collection,intervention_name,intervention_id,protocol_id,source_subject_id,sex,primary_diagnosis,brain_region,region_level_1,region_level_2,region_level_3,sample
0,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
1,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
2,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
3,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
4,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1


In [15]:
df.brain_region.value_counts()

brain_region
Middle temporal gyrus    461
IPL                      426
Substantia nigra         408
ACG                      402
Prefrontal cortex        252
Putamen                  222
Middle frontal gyrus     204
Hippocampus              204
Amygdala                 204
Substantia Nigra         168
Cingulate Cortex         114
Prefrontal Cortex         72
Name: count, dtype: int64

In [16]:
# get just the substantia nigra samples
df_nigra = df[df["brain_region"] == "Substantia nigra"]
df

,ASAP_sample_id,ASAP_subject_id,ASAP_team_id,ASAP_dataset_id,replicate,condition_id,age_at_collection,intervention_name,intervention_id,protocol_id,source_subject_id,sex,primary_diagnosis,brain_region,region_level_1,region_level_2,region_level_3,sample
0,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
1,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
2,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
3,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
4,ASAP_PMBDS_000026_s001,ASAP_PMBDS_000026,TEAM_HAFLER,DS_PMDBS_0002,Rep1,Control,81.0,Case-Control,Control,NaN,hSDG07,Female,Healthy Control,Prefrontal Cortex,Frontal lobe,Prefrontal cortex,Grey matter,ASAP_PMBDS_000026_s001_Rep1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3132,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN,T-5547,Female,Other neurological disorder,Substantia Nigra,Midbrain,Substantia nigra,Grey matter,ASAP_PMBDS_000230_s001_Rep1
3133,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN,T-5547,Female,Other neurological disorder,Substantia Nigra,Midbrain,Substantia nigra,Grey matter,ASAP_PMBDS_000230_s001_Rep1
3134,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN,T-5547,Female,Other neurological disorder,Substantia Nigra,Midbrain,Substantia nigra,Grey matter,ASAP_PMBDS_000230_s001_Rep1
3135,ASAP_PMBDS_000230_s001,ASAP_PMBDS_000230,TEAM_SULZER,DS_PMDBS_0006,Rep1,Control,89.0,Case-Control,Control,NaN,T-5547,Female,Other neurological disorder,Substantia Nigra,Midbrain,Substantia nigra,Grey matter,ASAP_PMBDS_000230_s001_Rep1


In [19]:
df_nigra["ASAP_team_id"].value_counts()

ASAP_team_id
TEAM_LEE          204
TEAM_JAKOBSSON    204
Name: count, dtype: int64

In [17]:
# Recode brain region to be "PFC", "MFG", "HIP", "SN", "ACG", "IPL, "AMG", "PUT"
brain_fix = {
    "Prefrontal Cortex": "PFC",
    "Middle_Frontal_Gyrus": "MFG",
    "Hippocampus": "HIP",
    "Substantia_Nigra ": "SN",
    "ACG": "ACG",
    "IPL": "IPL",
    "Middle temporal gyrus": "MTG",
    "Substantia nigra": "SN",
    "Prefrontal cortex": "PFC",
    "Amygdala": "AMG",
    "Putamen": "PUT",
}
df["brain_region"] = df["brain_region"].map(brain_fix)

In [18]:
# Map to find more course designations
brain_simple = {
    "PFC": "frontal_ctx",
    "MFG": "frontal_ctx",
    "ACG": "cingulate_ctx",
    "IPL": "parietal_ctx",
    "MTG": "temporal_ctx",
    "HIP": "subcortical",
    "AMG": "subcortical",
    "PUT": "subcortical",
    "SN": "subcortical",
}

df["brain_region_simple"] = df["brain_region"].map(brain_simple)


# Define sample to match
br_mapper_full = dict(zip(df["sample"], df["brain_region"]))
br_mapper_simple = dict(zip(df["sample"], df["brain_region"].map(brain_simple)))

# Parkinsons and control samples
condition_id_mapper = dict(zip(df["sample"], df["condition_id"]))
case_id_mapper = dict(zip(df["sample"], df["intervention_name"]))

# Detailed brain region mapper
region_1_mapper = dict(zip(df["sample"], df["region_level_1"]))
region_2_mapper = dict(zip(df["sample"], df["region_level_2"]))

# Diagnoses
diagnoses_mapper = dict(zip(df["sample"], df["primary_diagnosis"]))

In [53]:
dataset_metadata_filen = local_data_path / "asap-cohort-dataset-metadata.csv"
df.to_csv(dataset_metadata_filen)

In [58]:
local_data_path

PosixPath('/home/ergonyc/workspace/ws_files/data')

In [54]:
cell_metadata = adata.obs.copy()

In [55]:
# # Map samples to metadata
# adata.obs["brain_region"] = adata.obs["sample"].map(br_mapper_full)
# adata.obs["brain_region_simple"] = adata.obs["sample"].map(br_mapper_simple)
# adata.obs["case_id"] = adata.obs["sample"].map(case_id_mapper)
# adata.obs["condition_id"] = adata.obs["sample"].map(condition_id_mapper)
# adata.obs["region_level_1"] = adata.obs["sample"].map(region_1_mapper)
# adata.obs["region_level_2"] = adata.obs["sample"].map(region_2_mapper)

# Map samples to metadata
cell_metadata["brain_region"] = cell_metadata["sample"].map(br_mapper_full)
cell_metadata["brain_region_simple"] = cell_metadata["sample"].map(br_mapper_simple)
cell_metadata["case_id"] = cell_metadata["sample"].map(case_id_mapper)
cell_metadata["condition_id"] = cell_metadata["sample"].map(condition_id_mapper)
cell_metadata["region_level_1"] = cell_metadata["sample"].map(region_1_mapper)
cell_metadata["region_level_2"] = cell_metadata["sample"].map(region_2_mapper)

## Subset for Substantia Nigra + Neuronal Cells

In [56]:
# identify substantia nigra cells
sn_cells = cell_metadata["brain_region"] == "SN"
# Final boolean mask for subsetting
include = sn_cells

In [57]:
include

TCATTGTCAACAGTGG-1_ASAP_PMBDS_000038_s001_Rep1.cleaned_unfiltered.h5ad    False
ATTGGGTTCCAACACA-1_ASAP_PMBDS_000038_s001_Rep1.cleaned_unfiltered.h5ad    False
TTGAACGGTGCCTACG-1_ASAP_PMBDS_000038_s001_Rep1.cleaned_unfiltered.h5ad    False
TTCATTGAGGCCCACT-1_ASAP_PMBDS_000038_s001_Rep1.cleaned_unfiltered.h5ad    False
GTCTGTCGTCACCGCA-1_ASAP_PMBDS_000038_s001_Rep1.cleaned_unfiltered.h5ad    False
                                                                          ...  
TCCGAAAAGATTGCGG-1_ASAP_PMBDS_000230_s001_Rep1.cleaned_unfiltered.h5ad    False
TGAATGCGTGCATTAC-1_ASAP_PMBDS_000230_s001_Rep1.cleaned_unfiltered.h5ad    False
TAACCAGCATCATGAC-1_ASAP_PMBDS_000230_s001_Rep1.cleaned_unfiltered.h5ad    False
GTAACACCACCCTTGT-1_ASAP_PMBDS_000230_s001_Rep1.cleaned_unfiltered.h5ad    False
AGTCTCCCAACTCCAA-1_ASAP_PMBDS_000230_s001_Rep1.cleaned_unfiltered.h5ad    False
Name: brain_region, Length: 3578056, dtype: bool

In [ ]:
print(sum(include))

In [ ]:
# Create SN subset
sn_ad = adata[include].to_memory()
adata.file.close()  # close the original adata file

In [ ]:
#
og_obs = sn_ad.obs.copy()

new_obs = cell_metadata[include]

In [ ]:
new_obs.shape

In [ ]:
cell_metadata.shape, new_obs.shape, og_obs.shape

In [ ]:
sn_ad.obs = new_obs

In [ ]:
sn_ad.obs["condition_id"].value_counts()

### Export SN - Subset

In [ ]:
snn_samples_filename = local_data_path / f"asap-{dataset_team}.sn__samples.h5ad"
sn_ad.write_h5ad(snn_samples_filename)

In [ ]:
new_obs

In [ ]:
sn_ad.obs["phenotype"].value_counts()

keeping PD and healthy together for now due to little number of neuronal cells

In [ ]:
# Use regex to match multiple patterns
neuronal_subset = sn_ad[
    sn_ad.obs["class_name"].str.contains("GABAergic|Glutamatergic", regex=True)
].copy()

In [ ]:
neuronal_subset

### build a new raw dataset from the samples in sn_ad

In [ ]:
samples = sn_ad.obs["sample"]

In [ ]:
samples

In [ ]:
u_samples = list(samples.unique())
u_samples

In [ ]:
snn_samples_filename = local_data_path / f"asap-{dataset_team}.sn_neuronal_samples.h5ad"
neuronal_subset.write_h5ad(snn_samples_filename)

### Load Full Gene Expression for the Subset

In [ ]:
# Define file paths 
full_adata_filename = (
    cohort_analysis_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"
)
l_full_adata_filename = (
    local_data_path / f"asap-{dataset_team}.merged_cleaned_unfiltered.h5ad"
)

if not l_full_adata_filename.exists():
    !cp {full_adata_filename} {l_full_adata_filename}

In [ ]:
# Load full expression matrix
full_adata = sc.read_h5ad(l_full_adata_filename, backed="r")

In [ ]:
# Extract and select neuronal_subset cells from complete gene expression matrix
var_ = full_adata.var.copy()
X = full_adata[neuronal_subset.obs_names].X.copy()

full_adata.file.close()

combine the full gene expression matrix with our substantia nigra neuron subset, and save the resulting AnnDataobject.

In [ ]:
sn_neuronal_full_ad = sc.AnnData(
    X=X,
    obs=neuronal_subset.obs,
    var=var_,
    uns=neuronal_subset.uns,
    obsm=neuronal_subset.obsm,
)
sn_neuronal_full_ad

In [ ]:
# Save full sn neuronal Anndata object
sn_neuronal_full_samples_filename = (
    local_data_path / f"asap-{dataset_team}.full_sn_neuronal_samples.h5ad"
)
sn_neuronal_full_ad.write_h5ad(sn_neuronal_full_samples_filename)

## Analysis 

### Using marker genes: 
- [TH](https://www.genecards.org/cgi-bin/carddisp.pl?gene=TH&keywords=TH)
- [DDC](https://www.genecards.org/cgi-bin/carddisp.pl?gene=DDC&keywords=DDC)
- [SLC6A3](https://www.genecards.org/cgi-bin/carddisp.pl?gene=SLC6A3&keywords=SLC6A3)
- [NR4A2](https://www.genecards.org/cgi-bin/carddisp.pl?gene=NR4A2)
- [SLC18A2](https://www.genecards.org/cgi-bin/carddisp.pl?gene=SLC18A2&keywords=SLC18A2)


Together, these genes capture three essential aspects of the dopamine neuron phenotype:
1. Dopamine synthesis — TH, DDC
2. Dopamine storage and synaptic handling — SLC6A3, SLC18A2
3. Transcriptional identity and maintenance — NR4A2

Resources: 
- [https://www.nature.com/articles/s41593-022-01061-1](https://www.nature.com/articles/s41593-022-01061-1)
- [https://www.cell.com/cell-reports/references/S2211-1247(14)00862-6](https://www.cell.com/cell-reports/references/S2211-1247(14)00862-6)
- [https://www.pnas.org/doi/10.1073/pnas.2410331121](https://www.pnas.org/doi/10.1073/pnas.2410331121)

In [ ]:
# validate presence of marker genes
marker_genes = ["TH", "DDC", "SLC6A3", "NR4A2", "SLC18A2"]
for g in marker_genes:
    assert g in sn_neuronal_full_ad.var_names, f"{g} not found in var_names!"
existing_markers = [g for g in marker_genes if g in sn_neuronal_full_ad.var_names]

In [ ]:
# work on tmp copy
adata_neurons_tmp = sn_neuronal_full_ad.copy()

# recluster with only SN neurons
sc.pp.neighbors(adata_neurons_tmp, use_rep="X_scVI", n_neighbors=15)
sc.tl.leiden(adata_neurons_tmp, resolution=1.0, key_added="leiden_sn_scVI")

# normalize raw counts (CP10k)
sc.pp.normalize_total(adata_neurons_tmp, target_sum=1e4)

# log-transform
sc.pp.log1p(adata_neurons_tmp)

# score genes on normalized dataset
sc.tl.score_genes(adata_neurons_tmp, existing_markers, score_name="DAcore_score")

# compute mean gene expression / DA score for new clusters
cluster_means = (
    adata_neurons_tmp.obs.groupby("leiden_sn_scVI", observed=True)[["DAcore_score"]]
    .mean()
    .sort_values("DAcore_score", ascending=False)
)
print(cluster_means)


sc.pl.umap(
    adata_neurons_tmp,
    cmap=sns.cubehelix_palette(dark=0, light=0.9, as_cmap=True),
    color="leiden_sn_scVI",
)

In [ ]:
# validating k
for K in [10, 15, 20]:
    print(f"\n=== n_neighbors = {K} ===")
    sc.pp.neighbors(adata_neurons_tmp, use_rep="X_scVI", n_neighbors=K)
    sc.tl.leiden(adata_neurons_tmp, resolution=1.0, key_added=f"leiden_K{K}")

    sc.pl.violin(
        adata_neurons_tmp,
        keys=["DAcore_score", "TH", "DDC", "SLC6A3"],
        groupby=f"leiden_K{K}",
        rotation=90,
        stripplot=False,
    )

In [ ]:
adata_neurons_tmp.obs["leiden_sn_scVI"].value_counts()

In [ ]:
da_cluster = "11"
adata_DA = adata_neurons_tmp[
    adata_neurons_tmp.obs["leiden_sn_scVI"] == da_cluster
].copy()
adata_DA

26 DA cells too small for subtyping -- will consider this highly confident (Tier 1) 
--> potentially could broaden filter 

## prelimary data visulizations for understanding (ignore)

In [ ]:
# compute mean expression per group (normalized data)
group_means = (
    adata_neurons_tmp[:, existing_markers]
    .to_df()
    .groupby(adata_neurons_tmp.obs["supertype_name"], observed=True)
    .mean()
)

# find groups with all zero expression
empty_groups = group_means[(group_means == 0).all(axis=1)].index.tolist()
print("Groups with no expression:", empty_groups)

# subset AnnData to exclude those groups
filtered_ad = adata_neurons_tmp[
    ~adata_neurons_tmp.obs["supertype_name"].isin(empty_groups)
].copy()

# plot dotplot on filtered object
sc.pl.dotplot(
    filtered_ad,
    var_names=existing_markers,
    groupby=["C_scANVI", "supertype_name"],
    standard_scale="var",
    title="Dotplot grouped by C_scANVI and supertype_name (filtered)",
    cmap="viridis",
)

In [ ]:
# # Compute mean expression per group
# df = adata_neurons_tmp[:, existing_markers].to_df()

# # Add grouping columns from obs
# df["C_scANVI"] = adata_neurons_tmp.obs["C_scANVI"].values
# df["supertype_name"] = adata_neurons_tmp.obs["supertype_name"].values

# group_means = df.groupby(["C_scANVI", "supertype_name"], observed=True).mean()
# # Compute number of cells per group
# group_counts = df.groupby(["C_scANVI", "supertype_name"], observed=True).size()

# # Combine into one table
# group_means_table = group_means.reset_index()
# group_means_table["n_cells"] = group_counts.values

# # Build mask: rows where ANY marker gene > 0
# mask = (group_means_table[marker_genes] > 0).any(axis=1)

# # Apply mask
# filtered_table = group_means_table[mask].sort_values(by="n_cells", ascending =False)

# print(filtered_table)

# DAmean_table_filename = (local_data_path / "DA_marker_means.csv")
# group_means_table.to_csv(DAmean_table_filename)

In [ ]:
sc.pl.umap(
    adata_neurons_tmp,
    color=existing_markers,
    cmap=sns.cubehelix_palette(dark=0, light=0.9, as_cmap=True),
    ncols=2,
)

In [ ]:
# Extract expression matrix for these genes
expr = adata_neurons_tmp[:, existing_markers].X

# Convert sparse to dense if needed
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()

# Count how many markers are expressed (>0) per cell
marker_counts = (expr > 0).sum(axis=1)

# Gate: cells with at least 2 markers expressed
mask = marker_counts >= 2
# Subset AnnData
adata_DA = adata_neurons_tmp[mask].copy()

# Annotate in obs
adata_neurons_tmp.obs["DA_candidate"] = mask

# Print how many cells satisfy this
print("Number of cells with ≥ 2 markers expressed:", mask.sum())

# Print also how many total cells
print("Total cells:", len(marker_counts))
print("Fraction:", mask.sum() / len(marker_counts))

In [ ]:
sc.pl.umap(adata_neurons_tmp, color=["DA_candidate", "DAcore_score"])

In [ ]:
unique, counts = np.unique(marker_counts, return_counts=True)
for u, c in zip(unique, counts):
    print(f"{u} markers: {c} cells")